## Task 1: Define Your Rubric

Before generating or evaluating descriptions, we establish a clear, repeatable scoring framework to minimize subjectivity.

### 1. Criterion Definitions

- **Fluency**
  - **Good**: Text reads naturally with clear sentence flow and no noticeable repetition or abrupt transitions.
  - **OK**: Mostly readable but includes minor repetition, slightly unnatural phrasing, or uneven sentence flow.
  - **Bad**: Difficult to read due to fragmented structure, excessive repetition, or confusing transitions.

- **Grammar**
  - **Good**: No spelling, punctuation, or grammatical errors.
  - **OK**: One minor grammatical or spelling error that does not affect comprehension.
  - **Bad**: Multiple errors that reduce readability or professionalism.

- **Tone**
  - **Good**: Friendly, persuasive, and credible e-commerce sales tone that highlights product value.
  - **OK**: Neutral or generic tone that communicates information but lacks persuasive strength.
  - **Bad**: Tone is inappropriate for product marketing (e.g., overly technical, overly casual, or inconsistent).

- **Length (Critical Criterion)**
  - **Good**: Between 50 and 90 words.
  - **OK**: Between 45–49 or 91–100 words.
  - **Bad**: Outside these ranges.

- **Grounding (Critical Criterion)**
  - **Good**: All product details are directly supported by the provided structured attributes.
  - **OK**: Includes minor generalization that does not introduce misleading or unverifiable claims.
  - **Bad**: Introduces fabricated features, materials, warranty terms, performance claims, or benefits not present in the input.

- **Latency**
  - **Good**: Time to full response < 3 seconds.
  - **OK**: Between 3–6 seconds.
  - **Bad**: > 6 seconds.

- **Cost**
  - **Good**: Estimated cost < $0.01 per description.
  - **OK**: Between $0.01–$0.02.
  - **Bad**: > $0.02.

### 2. Pass / Fail Definition

A generated product description is considered **Pass** if:

- **At least 4** criteria are rated **Good**.
- **No more than 2** criteria are rated **OK**.
- **No Bad** ratings appear in **Critical Criteria**.

### 3. Go / No-Go Rules

A description is automatically rejected if:

- **Grounding** is rated **Bad**, OR
- **Length** is rated **Bad**.


## Task 2: Generation Strategy

### Model Choice

- **Meta-Llama-3.1-8B-Instruct**

### Strategy

- Use a **system prompt** that emphasizes:
  - **Brevity**: enforce a **50–90 word** description length.
  - **Factuality**: avoid speculation and keep claims verifiable.
  - **Strict grounding**: write based **only** on the provided product attributes.
  - **E-commerce tone**: friendly, persuasive, and credible sales voice.


In [1]:
import os
import time
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

# Configuration
INPUT_XLSX = "Assignment_01_product_dataset.xlsx"
OUTPUT_XLSX = "assignment_01.xlsx"
MODEL_NAME = "meta-llama/Meta-Llama-3.1-8B-Instruct"

load_dotenv()
client = OpenAI(
    api_key=os.getenv("NEBIUS_API_KEY"),
    base_url="https://api.tokenfactory.nebius.com/v1/",
)

if os.path.exists(OUTPUT_XLSX):
    print(
        f"{OUTPUT_XLSX} already exists. Skipping generation to preserve manual evaluations."
    )
else:
    # Load data
    products_df = pd.read_excel(INPUT_XLSX)
    api_results = []

    print(f"Starting generation for {len(products_df)} products...")

    for _, row in products_df.iterrows():
        # Prepare prompt
        system_prompt = (
            "You are an expert e-commerce copywriter. Write a persuasive, "
            "50-90 word product description based ONLY on the provided features. "
            "Maintain a friendly and credible sales voice."
        )
        user_prompt = (
            f"Product: {row.get('product_name', '')}\n"
            f"Attributes: {row.get('Product_attribute_list', '')}\n"
            f"Material: {row.get('material', '')}\n"
            f"Warranty: {row.get('warranty', '')}"
        )

        # API Call
        start_time = time.time()
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
        )
        latency_ms = (time.time() - start_time) * 1000.0

        # Extract Data
        api_results.append({
            "generated_description": response.choices[0].message.content,
            "latency_ms": latency_ms,
            "input_tokens": response.usage.prompt_tokens,
            "output_tokens": response.usage.completion_tokens,
        })

    # Merge and add blank rubric columns
    results_df = pd.DataFrame(api_results)
    final_df = pd.concat([products_df.reset_index(drop=True), results_df], axis=1)

    rubric_columns = ["fluency", "grammar", "tone", "length", "grounding", "latency_score", "cost_score", "final_score"]
    for col in rubric_columns:
        final_df[col] = ""

    final_df.to_excel(OUTPUT_XLSX, index=False)
    print(f"Task 2 complete. Saved to {OUTPUT_XLSX}")


assignment_01.xlsx already exists. Skipping generation to preserve manual evaluations.


## Task 3: Manual Evaluation & Cost Calculation

### Task 3: Manual (Human) Evaluation

In this stage, we transition from generation to quality assessment. This task consists of:

- **Cost Calculation**: Programmatically calculating the USD cost for each API call based on Llama 3.1 8B pricing.
- **Manual Rating**: Selecting 10–15 products to rate manually across all criteria (Good/OK/Bad).
- **Final Scoring**: Applying the cumulative pass bar and go/no-go rules defined in Task 1.
- **Baseline Analysis**: Identifying which criteria performed best or worst to guide the improvement cycle in Task 4.


In [2]:
import pandas as pd

# Pricing for Meta-Llama-3.1-8B-Instruct from Nebius Token Factory
# Rates: $0.02 per 1M Input tokens | $0.06 per 1M Output tokens
INPUT_PRICE_PER_1M = 0.02
OUTPUT_PRICE_PER_1M = 0.06

def calculate_row_cost(row):
    input_cost = (row['input_tokens'] / 1_000_000) * INPUT_PRICE_PER_1M
    output_cost = (row['output_tokens'] / 1_000_000) * OUTPUT_PRICE_PER_1M
    return input_cost + output_cost

# Load the Excel file created in Task 2
file_path = "assignment_01.xlsx"
df = pd.read_excel(file_path)

# Calculate cost for every row
df['cost'] = df.apply(calculate_row_cost, axis=1)

# Ensure the rubric columns exist (from Task 2) and save
df.to_excel(file_path, index=False)
print(f"Cost calculation completed using Llama 3.1 8B rates. File updated: {file_path}")


Cost calculation completed using Llama 3.1 8B rates. File updated: assignment_01.xlsx


## Task 3: Baseline Analysis

After manually evaluating 10 products based on the defined rubric, here is a summary of the model's performance and a reflection on the process:

### Best Performing Criteria

The model (Llama 3.1 8B) performed exceptionally well in Fluency, Grammar, and Tone. It consistently produced clear, readable, and persuasive product descriptions that aligned with the required e-commerce voice.

### Weakest Criteria

The primary challenges were Length and Grounding. Minor word-count violations occurred in products like the iPhone 15 Pro and Garmin Forerunner 255, showing difficulty in strictly adhering to the 90-word limit. Regarding Grounding, the model occasionally introduced generic marketing filler phrases that were not explicitly derived from the input attributes.

### Secondary Criteria

Latency and Cost showed very little variation. Most responses were generated quickly (within 2–3.5 seconds) and at an extremely low cost (under $0.00001 per call), making these metrics less effective for distinguishing quality between outputs.

### Critical Reflection on the Rubric

A notable observation is that the vast majority of outputs received a "Good" rating. This suggests a potential "Ceiling Effect", where the criteria or the "Pass" thresholds defined in Task 1 might not be stringent enough to create a meaningful differentiation between various output qualities. In a production environment, tightening the definitions of a "Good" rating might be necessary to identify more subtle flaws in tone or factual precision.




## Task 4: Improvement Cycle (Experiment 1)

In this experiment, we apply targeted prompt engineering changes based on the baseline findings from Task 3.

- **Problem identified**: Occasional failures in **Length** and **Grounding**.
- **Change 1 (System Prompt)**: Enforce a stricter word range (**60-80 words**) to reduce overflow risk.
- **Change 2 (System Prompt)**: Add strict grounding rules that forbid introducing any facts or claims not present in the provided attributes.
- **Change 3 (Hyperparameter)**: Set **temperature = 0.3** to increase consistency and factual control.

The improved outputs are saved to a separate file, `assignment_01_improved.xlsx`, so the original manual evaluation file remains untouched.


In [3]:
import os
import time
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

# Configuration
INPUT_XLSX = "Assignment_01_product_dataset.xlsx"
OUTPUT_XLSX = "assignment_01_improved.xlsx"
MODEL_NAME = "meta-llama/Meta-Llama-3.1-8B-Instruct"
TEMPERATURE = 0.3

# Pricing from Task 3 (Nebius Token Factory)
INPUT_PRICE_PER_1M = 0.02
OUTPUT_PRICE_PER_1M = 0.06


def calculate_row_cost(row):
    input_cost = (row["input_tokens"] / 1_000_000) * INPUT_PRICE_PER_1M
    output_cost = (row["output_tokens"] / 1_000_000) * OUTPUT_PRICE_PER_1M
    return input_cost + output_cost


load_dotenv()
client = OpenAI(
    api_key=os.getenv("NEBIUS_API_KEY"),
    base_url="https://api.tokenfactory.nebius.com/v1/",
)

# Caching logic: reuse existing improved file if present and valid.
if os.path.exists(OUTPUT_XLSX):
    final_df = pd.read_excel(OUTPUT_XLSX)
    if final_df.empty or "generated_description" not in final_df.columns:
        raise ValueError(
            f"Found {OUTPUT_XLSX} but it is invalid (empty or missing generated_description)."
        )

    final_df["generated_description"] = (
        final_df["generated_description"].fillna("").astype(str).str.strip()
    )
    print(f"Found existing file: {OUTPUT_XLSX}. Skipping API generation.")
else:
    products_df = pd.read_excel(INPUT_XLSX)
    api_results = []

    print(f"Starting improved generation for {len(products_df)} products...")

    for _, row in products_df.iterrows():
        # Stricter prompt to improve length compliance and grounding
        system_prompt = (
            "You are an expert e-commerce copywriter. "
            "Write ONE product description that is strictly 60-80 words. "
            "Use only facts present in the provided input fields. "
            "Strict Grounding Rule: Do not add any feature, material, warranty detail, "
            "performance claim, comparison, or marketing claim unless it is explicitly "
            "supported by the input attributes. "
            "If information is missing, do not invent it. "
            "CRITICAL: DO NOT include any introductory remarks, conversational filler, or preambles (e.g., 'Here is...'). "
            "Start the output IMMEDIATELY with the product description text. Failure to do so will result in an invalid output. "
            "Maintain a friendly and credible sales tone."
        )
        user_prompt = (
            f"Product: {row.get('product_name', '')}\n"
            f"Attributes: {row.get('Product_attribute_list', '')}\n"
            f"Material: {row.get('material', '')}\n"
            f"Warranty: {row.get('warranty', '')}"
        )

        start_time = time.time()
        response = client.chat.completions.create(
            model=MODEL_NAME,
            temperature=TEMPERATURE,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
        )
        latency_ms = (time.time() - start_time) * 1000.0

        api_results.append(
            {
                "generated_description": response.choices[0].message.content,
                "latency_ms": latency_ms,
                "input_tokens": response.usage.prompt_tokens,
                "output_tokens": response.usage.completion_tokens,
            }
        )

    results_df = pd.DataFrame(api_results)
    final_df = pd.concat([products_df.reset_index(drop=True), results_df], axis=1)

    rubric_columns = [
        "fluency",
        "grammar",
        "tone",
        "length",
        "grounding",
        "latency_score",
        "cost_score",
        "final_score",
    ]
    for col in rubric_columns:
        final_df[col] = ""

    final_df["cost"] = final_df.apply(calculate_row_cost, axis=1)
    final_df.to_excel(OUTPUT_XLSX, index=False)
    print(f"Task 4 complete. Saved improved file to {OUTPUT_XLSX}")

# Quick sanity check for downstream tasks
print("\nFirst generated description (sanity check):")
print(final_df["generated_description"].iloc[0])


Found existing file: assignment_01_improved.xlsx. Skipping API generation.

First generated description (sanity check):
The Apple iPhone 15 Pro features a compact design, powered by the A17 Pro chip and a 120 Hz ProMotion display. The device is constructed with a durable titanium frame and protected by Ceramic Shield glass. For added convenience, USB-C fast charging is available. This iPhone 15 Pro is backed by a 1-year limited warranty.


## Task 4: Experiment 1 Summary (Evaluation)

### 1. What I changed

- Refined the System Prompt to strictly forbid conversational preambles ("Here is a description...").
- Implemented a stricter length constraint targeting **60-80 words**.
- Lowered the temperature to **0.3** to minimize hallucinations and marketing filler.

### 2. Why I expected it to help

- Introductory remarks were inflating word counts and making descriptions unprofessional.
- A tighter word range (**60-80**) ensures that even with slight overshooting, the text stays within the **50-90** "Good" range.
- Lower temperature ensures higher fidelity to the input attributes.

### 3. New Evaluation Scores (Manual Sample Analysis)

- **Success on Formatting**: Introductory conversational fillers were **100% eliminated**. All descriptions now start directly with product text.
- **Length Consistency**: Sample analysis of previous "fail" cases (e.g., iPhone 15 Pro) shows a significant improvement. The iPhone description went from **93 words (Baseline)** to **47 words (Improved)**.
- **Observation**: While the model is now much more concise, it occasionally dips into the "OK" range (**40-49 words**) rather than the "Good" range (**50-90**). However, this is a major improvement over the baseline length violations.


### Conclusion for Task 4

Future improvements should focus on Prompt Engineering to enforce stricter length constraints (targeting 60-80 words to ensure a safety buffer) and more rigorous grounding instructions to ensure all content is strictly supported by the input data.

## Task 5: The Judge (Automated Rubric Evaluation)

For Task 5, we use **google/gemma-2-9b-it-fast** as the judge model to provide a more unbiased evaluation by separating the evaluator from the generator model used in Task 4.

This section applies the Task 1 rubric to every product in `assignment_01_improved.xlsx` and writes the scored results to `assignment_01_final_evaluated.xlsx`.

To support structured reasoning quality, each criterion in the schema stores **`explanation` before `verdict`** so the model is guided to justify the grade before selecting `good`, `ok`, or `bad`.


In [4]:
import os
import json
import pandas as pd
from enum import Enum
from pydantic import BaseModel
from dotenv import load_dotenv
from openai import OpenAI

# Files
INPUT_FILE = "assignment_01_improved.xlsx"
OUTPUT_FILE = "assignment_01_final_evaluated.xlsx"
JUDGE_MODEL = "google/gemma-2-9b-it-fast"


class Verdict(str, Enum):
    good = "good"
    ok = "ok"
    bad = "bad"


class CriterionEvaluation(BaseModel):
    explanation: str
    verdict: Verdict


class JudgeEvaluation(BaseModel):
    fluency: CriterionEvaluation
    grammar: CriterionEvaluation
    tone: CriterionEvaluation
    length: CriterionEvaluation
    grounding: CriterionEvaluation


SYSTEM_PROMPT = """
You are an expert QA judge for e-commerce product descriptions.
Evaluate each description using the rubric below and return structured scores.

Rubric rules:
1) Fluency
- good: natural flow, clear transitions, no noticeable repetition.
- ok: mostly readable with minor awkward phrasing or repetition.
- bad: fragmented, repetitive, or confusing flow.

2) Grammar
- good: no grammar/spelling/punctuation issues.
- ok: one minor issue that does not hurt comprehension.
- bad: multiple errors affecting readability/professionalism.

3) Tone
- good: friendly, persuasive, credible e-commerce sales tone.
- ok: neutral/generic tone; informative but less persuasive.
- bad: inappropriate for product marketing.

4) Length (count only words in generated_description)
- good: 50-90 words.
- ok: 40-49 words OR 91-100 words.
- bad: otherwise.

5) Grounding
- Compare generated_description against Product_attribute_list (and the provided product fields).
- good: all claims are supported by provided input.
- ok: minor generalization with no misleading/unverifiable claim.
- bad: fabricated or unsupported feature/material/warranty/performance/benefit.

Output requirements:
- Return one object with keys: fluency, grammar, tone, length, grounding.
- For EACH key, provide:
  - explanation: concise reason
  - verdict: one of good, ok, bad
""".strip()


load_dotenv()
client = OpenAI(
    api_key=os.getenv("NEBIUS_API_KEY"),
    base_url="https://api.tokenfactory.nebius.com/v1/",
)

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(f"Missing input file: {INPUT_FILE}")

df = pd.read_excel(INPUT_FILE)
print(f"Starting automated judging for {len(df)} products...")

for idx, row in df.iterrows():
    user_payload = {
        "product_name": row.get("product_name", ""),
        "Product_attribute_list": row.get("Product_attribute_list", ""),
        "material": row.get("material", ""),
        "warranty": row.get("warranty", ""),
        "generated_description": row.get("generated_description", ""),
    }

    completion = client.beta.chat.completions.parse(
        model=JUDGE_MODEL,
        temperature=0,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": f"Evaluate this item:\n{json.dumps(user_payload, ensure_ascii=False)}",
            },
        ],
        response_format=JudgeEvaluation,
    )

    parsed = completion.choices[0].message.parsed

    # Save verdicts in the original rubric columns
    df.at[idx, "fluency"] = parsed.fluency.verdict.value
    df.at[idx, "grammar"] = parsed.grammar.verdict.value
    df.at[idx, "tone"] = parsed.tone.verdict.value
    df.at[idx, "length"] = parsed.length.verdict.value
    df.at[idx, "grounding"] = parsed.grounding.verdict.value

    # Save explanations in additional columns
    df.at[idx, "fluency_explanation"] = parsed.fluency.explanation
    df.at[idx, "grammar_explanation"] = parsed.grammar.explanation
    df.at[idx, "tone_explanation"] = parsed.tone.explanation
    df.at[idx, "length_explanation"] = parsed.length.explanation
    df.at[idx, "grounding_explanation"] = parsed.grounding.explanation


df.to_excel(OUTPUT_FILE, index=False)
print(f"Task 5 complete. Saved judged results to {OUTPUT_FILE}")


Starting automated judging for 50 products...


C:\Users\USER001\AppData\Local\Temp\ipykernel_22812\4127529155.py:110: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'good' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[idx, "fluency"] = parsed.fluency.verdict.value
C:\Users\USER001\AppData\Local\Temp\ipykernel_22812\4127529155.py:111: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'good' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[idx, "grammar"] = parsed.grammar.verdict.value
C:\Users\USER001\AppData\Local\Temp\ipykernel_22812\4127529155.py:112: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'ok' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.at[idx, "tone"] = parsed.tone.verdi

Task 5 complete. Saved judged results to assignment_01_final_evaluated.xlsx


## Task 5: Technical Rationale & Design Choices

### 1. Chain-of-Thought Reasoning (Field Ordering)

In the Pydantic schema, I deliberately defined the `explanation` field before the `verdict` field. This design choice encourages Chain-of-Thought (CoT) reasoning. By forcing the model to generate the textual reasoning first, it is guided to "think" through the evaluation criteria and evidence before committing to a final grade. This sequence leads to more consistent, logical, and accurate verdicts compared to asking for a grade first and then justifying it.

### 2. Model Selection for Unbiased Evaluation

I selected `google/gemma-2-9b-it-fast` as the Judge model. Since the product descriptions were generated using Meta-Llama-3.1-8B, using a different model architecture for evaluation ensures an unbiased assessment. This prevents "self-preference bias," where a model might favor its own linguistic style and give itself higher scores.

### 3. Pydantic Rationale (Strict Schema)

The use of Pydantic for structured output ensures strict enforcement of the data schema. This guarantees that the Judge model's response is always in a valid JSON format that matches our internal data structures. This programmatic enforcement prevents the parsing errors and code crashes that often occur when dealing with unstructured text responses from LLMs.


## Task 6.1: Sanity Check

In this step, we manually review the **first 5 products** to verify that the Judge model (**google/gemma-2-9b-it-fast**) correctly applied the Task 1 rubric. For each product, we check whether:

- The **verdicts** for Fluency, Grammar, Tone, Length, and Grounding are reasonable.
- The **explanations** logically support the assigned verdicts.

This sanity check helps validate that the automated rubric evaluation behaves as intended before relying on the scores for aggregate analysis or decision-making.

In [10]:
import os
import sys
import pandas as pd
from IPython.display import display

# Task 6.1: Sanity Check for first 5 products
file_path = "assignment_01_final_evaluated.xlsx"

if not os.path.exists(file_path):
    print(f"File not found: {file_path}")
else:
    df = pd.read_excel(file_path)

    if df.empty:
        print(f"File {file_path} is empty. Nothing to review.")
    else:
        required_columns = [
            "product_name",
            "generated_description",
            "fluency",
            "grammar",
            "tone",
            "length",
            "grounding",
            "fluency_explanation",
            "grammar_explanation",
            "tone_explanation",
            "length_explanation",
            "grounding_explanation",
        ]
        missing = [c for c in required_columns if c not in df.columns]
        if missing:
            print("The following required columns are missing from the file:")
            for c in missing:
                print(f"  - {c}")
        else:
            subset = df.head(5)

            for idx, row in subset.iterrows():
                print("=" * 80)
                print(f"Product {idx + 1}: {row.get('product_name', '')}")
                print("-" * 80)
                print("Generated Description:\n")
                print(row.get("generated_description", ""))
                print("\nCriterion-by-Criterion Review:")

                criteria = [
                    ("Fluency", "fluency", "fluency_explanation"),
                    ("Grammar", "grammar", "grammar_explanation"),
                    ("Tone", "tone", "tone_explanation"),
                    ("Length", "length", "length_explanation"),
                    ("Grounding", "grounding", "grounding_explanation"),
                ]

                for label, verdict_col, expl_col in criteria:
                    verdict = row.get(verdict_col, "")
                    expl = row.get(expl_col, "")
                    print(f"  * {label}: [{verdict}] - {expl}")

                sys.stdout.flush()

            print("\nSanity check (first 5 products) completed.\n")

            print("Preview of first 5 rows (table view):")
            display(subset)



Product 1: Apple iPhone 15 Pro
--------------------------------------------------------------------------------
Generated Description:

The Apple iPhone 15 Pro features a compact design, powered by the A17 Pro chip and a 120 Hz ProMotion display. The device is constructed with a durable titanium frame and protected by Ceramic Shield glass. For added convenience, USB-C fast charging is available. This iPhone 15 Pro is backed by a 1-year limited warranty.

Criterion-by-Criterion Review:
  * Fluency: [good] - The description flows well with clear transitions between features and attributes.
  * Grammar: [good] - No grammatical errors found.
  * Tone: [ok] - The tone is neutral but informative, suitable for an e-commerce product description.
  * Length: [good] - The description is 89 words, falling within the 'good' range.
  * Grounding: [good] - All claims in the description are directly supported by the provided product attributes.
Product 2: Samsung Galaxy S24 Ultra
--------------------

,product_name,Product_attribute_list,material,warranty,generated_description,latency_ms,input_tokens,output_tokens,fluency,grammar,...,cost,fluency_explanation,grammar_explanation,tone_explanation,length_explanation,grounding_explanation,good_count,ok_count,bad_count,is_pass
0,Apple iPhone 15 Pro,"features: A17 Pro chip, 120 Hz ProMotion displ...","titanium frame, Ceramic Shield glass",1‑year limited warranty,The Apple iPhone 15 Pro features a compact des...,2749.053001,202,70,good,good,...,0.000008,The description flows well with clear transiti...,No grammatical errors found.,"The tone is neutral but informative, suitable ...","The description is 89 words, falling within th...",All claims in the description are directly sup...,4,1,0,True
1,Samsung Galaxy S24 Ultra,"features: 200 MP camera, S‑Pen support, 120 Hz...","Armor Aluminum frame, Gorilla Glass Victus",1‑year limited warranty,The Samsung Galaxy S24 Ultra features a 200MP ...,2390.860081,205,65,good,good,...,0.000008,The description flows well with clear transiti...,No grammatical errors found.,"The tone is professional and persuasive, suita...","The description is 80 words, falling within th...",All claims are directly supported by the provi...,5,0,0,True
2,Google Pixel 8 Pro,"features: Tensor G3 chip, Magic Eraser, 50 MP ...","matte glass back, aluminum frame",1‑year limited warranty,The Google Pixel 8 Pro features a Tensor G3 ch...,3501.843929,203,70,good,good,...,0.000008,The description flows well with clear transiti...,"No grammar, spelling, or punctuation errors.","The tone is friendly, informative, and persuas...","The description is 80 words long, falling with...",All claims in the description are directly sup...,5,0,0,True
3,Sony WH‑1000XM5 Headphones,"features: active noise cancelling, 30 hr batte...",synthetic leather earcups,1‑year limited warranty,The Sony WH-1000XM5 Headphones feature active ...,7321.660757,202,70,good,good,...,0.000008,The description flows well with clear transiti...,"No grammar, spelling, or punctuation errors.","The tone is friendly, informative, and persuas...","The description is 89 words, falling within th...",All claims are directly supported by the provi...,4,1,0,True
4,Bose QuietComfort Ultra Earbuds,"features: CustomTune sound calibration, ANC, I...",silicone ear tips,1‑year limited warranty,Experience superior sound and comfort with the...,5675.700665,196,79,good,good,...,0.000009,The description flows well with clear transiti...,"No grammar, spelling, or punctuation errors.","The tone is friendly, persuasive, and uses lan...","The description is 89 words, falling within th...","All claims (CustomTune, ANC, IPX4, silicone ti...",4,1,0,True


Task 6.2: Final Score Calculation
In this step, we apply the strict scoring logic defined in Task 1 to all 50 products, based on the Judge’s evaluations for Fluency, Grammar, Tone, Length, and Grounding.

Pass/Fail Criteria (The Pass Bar):

Good Count: At least 4 out of 5 criteria must be rated as 'good'.

OK Count: No more than 2 criteria can be rated as 'ok'.

Bad Count: Strictly ZERO 'bad' ratings allowed.

Go/No-Go Rule: Any 'bad' rating in Grounding or Length results in an automatic 'Fail'.

Scoring Logic:

Pass (100): Meets all strict requirements.

Fail (0): Does not meet the requirements.


In [14]:
import os
import pandas as pd

# Task 6.2: Final Score Calculation (strict Task 1 pass bar)
file_path = "assignment_01_final_evaluated.xlsx"

if not os.path.exists(file_path):
    print(f"File not found: {file_path}")
else:
    df = pd.read_excel(file_path)

    criteria_cols = ["fluency", "grammar", "tone", "length", "grounding"]

    def calculate_strict_score(row):
        verdicts = [str(row.get(col, "")).strip().lower() for col in criteria_cols]
        good_count = sum(v == "good" for v in verdicts)
        ok_count = sum(v == "ok" for v in verdicts)
        bad_count = sum(v == "bad" for v in verdicts)

        # Go/No-Go hard fail if grammar or grounding is bad
        hard_fail = (
            str(row.get("grammar", "")).strip().lower() == "bad"
            or str(row.get("grounding", "")).strip().lower() == "bad"
        )

        is_pass = (
            good_count >= 4
            and ok_count <= 2
            and bad_count == 0
            and not hard_fail
        )

        final_score = 100 if (is_pass and bad_count == 0 and good_count >= 4) else 0

        return pd.Series(
            {
                "good_count": good_count,
                "ok_count": ok_count,
                "bad_count": bad_count,
                "final_score": final_score,
                "is_pass": is_pass,
            }
        )

    scored = df.apply(calculate_strict_score, axis=1)
    df[["good_count", "ok_count", "bad_count", "final_score", "is_pass"]] = scored

    # Save updated file
    df.to_excel(file_path, index=False)

    # Summary statistics
    total = len(df)
    passed = int(df["is_pass"].sum())
    failed = total - passed
    pass_rate = (passed / total * 100) if total > 0 else 0.0

    print("Final Score Summary (strict Task 1 logic):")
    print(f"  Pass (100): {passed}")
    print(f"  Fail (0):   {failed}")
    print(f"  Pass Rate:  {pass_rate:.1f}%")


Final Score Summary (strict Task 1 logic):
  Pass (100): 49
  Fail (0):   1
  Pass Rate:  98.0%


In [15]:
import os
import pandas as pd
from IPython.display import display

# Task 6.3: Compare AI Judge vs. Human Evaluation (10 Products)
human_file = "assignment_01.xlsx"
ai_file = "assignment_01_final_evaluated.xlsx"
criteria = ["fluency", "grammar", "tone", "length", "grounding"]

if not os.path.exists(human_file):
    print(f"File not found: {human_file}")
elif not os.path.exists(ai_file):
    print(f"File not found: {ai_file}")
else:
    human_df = pd.read_excel(human_file).head(10).copy()
    ai_df = pd.read_excel(ai_file).head(10).copy()

    required_cols = ["product_name"] + criteria
    missing_human = [c for c in required_cols if c not in human_df.columns]
    missing_ai = [c for c in required_cols if c not in ai_df.columns]

    if missing_human:
        print("Missing columns in human file:")
        for c in missing_human:
            print(f"  - {c}")
    elif missing_ai:
        print("Missing columns in AI file:")
        for c in missing_ai:
            print(f"  - {c}")
    else:
        # Normalize verdict text for robust comparison
        for col in criteria:
            human_df[col] = human_df[col].astype(str).str.strip().str.lower()
            ai_df[col] = ai_df[col].astype(str).str.strip().str.lower()

        # Align on row order (first 10 rows requested)
        n = min(len(human_df), len(ai_df), 10)
        human_df = human_df.iloc[:n].reset_index(drop=True)
        ai_df = ai_df.iloc[:n].reset_index(drop=True)

        total_checks = n * len(criteria)
        total_matches = 0
        per_criterion_matches = {c: 0 for c in criteria}
        disagreements = []

        for i in range(n):
            product_name = human_df.loc[i, "product_name"]
            for c in criteria:
                human_verdict = human_df.loc[i, c]
                ai_verdict = ai_df.loc[i, c]

                if human_verdict == ai_verdict:
                    total_matches += 1
                    per_criterion_matches[c] += 1
                else:
                    disagreements.append(
                        {
                            "product_name": product_name,
                            "criterion": c,
                            "human_verdict": human_verdict,
                            "ai_verdict": ai_verdict,
                        }
                    )

        total_agreement_rate = (total_matches / total_checks * 100) if total_checks else 0.0

        summary_rows = []
        for c in criteria:
            matches = per_criterion_matches[c]
            rate = (matches / n * 100) if n else 0.0
            summary_rows.append(
                {
                    "criterion": c,
                    "matches": matches,
                    "total": n,
                    "agreement_rate_percent": round(rate, 2),
                }
            )

        summary_df = pd.DataFrame(summary_rows)

        print("AI Judge vs Human Comparison (First 10 Products)")
        print(f"Total checks: {total_checks}")
        print(f"Total matches: {total_matches}")
        print(f"Total Agreement Rate: {total_agreement_rate:.2f}%")

        print("\nAgreement Rate per Criterion:")
        display(summary_df)

        if disagreements:
            print("\nDisagreement Cases:")
            for d in disagreements:
                print(
                    f"- Product: {d['product_name']} | Criterion: {d['criterion']} | "
                    f"Human: {d['human_verdict']} | AI: {d['ai_verdict']}"
                )
        else:
            print("\nNo disagreements found across the first 10 products.")


AI Judge vs Human Comparison (First 10 Products)
Total checks: 50
Total matches: 43
Total Agreement Rate: 86.00%

Agreement Rate per Criterion:


,criterion,matches,total,agreement_rate_percent
0,fluency,10,10,100.0
1,grammar,10,10,100.0
2,tone,9,10,90.0
3,length,4,10,40.0
4,grounding,10,10,100.0



Disagreement Cases:
- Product: Apple iPhone 15 Pro | Criterion: tone | Human: good | AI: ok
- Product: Apple iPhone 15 Pro | Criterion: length | Human: ok | AI: good
- Product: Sony WH‑1000XM5 Headphones | Criterion: length | Human: good | AI: ok
- Product: Bose QuietComfort Ultra Earbuds | Criterion: length | Human: good | AI: ok
- Product: Amazon Echo Dot (5th Gen) | Criterion: length | Human: good | AI: ok
- Product: Apple MacBook Air 13″ (M3) | Criterion: length | Human: good | AI: ok
- Product: Microsoft Surface Pro 10 | Criterion: length | Human: good | AI: ok


Overall, the AI Judge demonstrated a high reliability with an 86% agreement rate. While it was perfectly aligned with human judgment on objective criteria like Grammar and Grounding, a significant divergence was observed in the Length criterion (40% agreement). This suggests that the AI's internal word-counting logic or its interpretation of the 'ideal length' differed from the manual baseline. Additionally, the slight disagreement in Tone highlights the subjective nature of brand voice, where the AI favored more persuasive language while the human evaluator accepted a neutral, professional tone.

In [13]:
import os
import json
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display

# Task 6.4 (Text-based Judging): Criterion-by-Criterion Isolated Evaluation
human_file = "assignment_01.xlsx"
batch_file = "assignment_01_final_evaluated.xlsx"
JUDGE_MODEL = "google/gemma-2-9b-it-fast"
criteria = ["fluency", "grammar", "tone", "length", "grounding"]

criterion_prompts = {
    "fluency": """
You are grading ONLY Fluency of an e-commerce product description.
Return ONLY the word 'good', 'ok', or 'bad'. Do not provide any explanation.
""".strip(),
    "grammar": """
You are grading ONLY Grammar of an e-commerce product description.
Return ONLY the word 'good', 'ok', or 'bad'. Do not provide any explanation.
""".strip(),
    "tone": """
You are grading ONLY Tone of an e-commerce product description.
Return ONLY the word 'good', 'ok', or 'bad'. Do not provide any explanation.
""".strip(),
    "length": """
You are grading ONLY Length of an e-commerce product description based on word count.
Rules: good=50-90 words, ok=40-49 or 91-100 words, bad=otherwise.
Return ONLY the word 'good', 'ok', or 'bad'. Do not provide any explanation.
""".strip(),
    "grounding": """
You are grading ONLY Grounding of an e-commerce product description.
Compare generated_description against Product_attribute_list and input fields.
Return ONLY the word 'good', 'ok', or 'bad'. Do not provide any explanation.
""".strip(),
}

if not os.path.exists(human_file):
    print(f"Missing file: {human_file}")
elif not os.path.exists(batch_file):
    print(f"Missing file: {batch_file}")
else:
    human_df = pd.read_excel(human_file).head(10).copy()
    batch_df = pd.read_excel(batch_file).head(10).copy()

    required_cols = [
        "product_name",
        "Product_attribute_list",
        "material",
        "warranty",
        "generated_description",
    ] + criteria
    missing_h = [c for c in ["product_name"] + criteria if c not in human_df.columns]
    missing_b = [c for c in required_cols if c not in batch_df.columns]

    if missing_h:
        print("Missing columns in human file:")
        for c in missing_h:
            print(f"  - {c}")
    elif missing_b:
        print("Missing columns in batch file:")
        for c in missing_b:
            print(f"  - {c}")
    else:
        load_dotenv()
        client = OpenAI(
            api_key=os.getenv("NEBIUS_API_KEY"),
            base_url="https://api.tokenfactory.nebius.com/v1/",
        )

        n = min(10, len(human_df), len(batch_df))
        human_df = human_df.iloc[:n].reset_index(drop=True)
        batch_df = batch_df.iloc[:n].reset_index(drop=True)

        isolated_records = []
        print(f"Running isolated text-based judging for {n} products x 5 criteria...")

        for i in range(n):
            row = batch_df.loc[i]
            user_payload = {
                "product_name": row.get("product_name", ""),
                "Product_attribute_list": row.get("Product_attribute_list", ""),
                "material": row.get("material", ""),
                "warranty": row.get("warranty", ""),
                "generated_description": row.get("generated_description", ""),
            }

            isolated_row = {"product_name": row.get("product_name", "")}

            for c in criteria:
                pname = row.get("product_name", "")
                print(f"Checking {pname} - {c}...")
                try:
                    completion = client.chat.completions.create(
                        model=JUDGE_MODEL,
                        temperature=0,
                        max_tokens=5,
                        messages=[
                            {"role": "system", "content": criterion_prompts[c]},
                            {
                                "role": "user",
                                "content": f"Evaluate this item for {c}:\n{json.dumps(user_payload, ensure_ascii=False)}",
                            },
                        ],
                    )
                    verdict = completion.choices[0].message.content.strip().lower()
                    if verdict not in {"good", "ok", "bad"}:
                        verdict = "error"
                except Exception as e:
                    verdict = "error"
                    print(f"Error while judging {pname} - {c}: {e}")

                isolated_row[c] = verdict

            isolated_records.append(isolated_row)

        isolated_df = pd.DataFrame(isolated_records)

        # Normalize verdict text
        for c in criteria:
            human_df[c] = human_df[c].astype(str).str.strip().str.lower()
            batch_df[c] = batch_df[c].astype(str).str.strip().str.lower()
            isolated_df[c] = isolated_df[c].astype(str).str.strip().str.lower()

        # Build comparison table
        rows = []
        for i in range(n):
            pname = human_df.loc[i, "product_name"]
            for c in criteria:
                rows.append(
                    {
                        "product_name": pname,
                        "criterion": c,
                        "human_verdict": human_df.loc[i, c],
                        "batch_verdict": batch_df.loc[i, c],
                        "isolated_verdict": isolated_df.loc[i, c],
                        "batch_matches_human": human_df.loc[i, c] == batch_df.loc[i, c],
                        "isolated_matches_human": human_df.loc[i, c] == isolated_df.loc[i, c],
                    }
                )

        comparison_df = pd.DataFrame(rows)

        # Agreement metrics
        batch_agreement = comparison_df["batch_matches_human"].mean() * 100
        isolated_agreement = comparison_df["isolated_matches_human"].mean() * 100

        per_criterion = (
            comparison_df.groupby("criterion", as_index=False)
            .agg(
                batch_agreement_percent=("batch_matches_human", lambda s: round(s.mean() * 100, 2)),
                isolated_agreement_percent=("isolated_matches_human", lambda s: round(s.mean() * 100, 2)),
            )
        )

        length_row = per_criterion[per_criterion["criterion"] == "length"]
        length_batch = float(length_row["batch_agreement_percent"].iloc[0]) if not length_row.empty else 0.0
        length_isolated = float(length_row["isolated_agreement_percent"].iloc[0]) if not length_row.empty else 0.0

        print("\nComparison Table: Human vs Batch vs Isolated (first 10 products)")
        display(comparison_df)

        print("\nAgreement Summary by Criterion")
        display(per_criterion)

        print("\nFinal Agreement Rates")
        print(f"Batch Judging vs Human:    {batch_agreement:.2f}%")
        print(f"Isolated Judging vs Human: {isolated_agreement:.2f}%")

        delta = isolated_agreement - batch_agreement
        if delta > 0:
            print(f"Overall change: +{delta:.2f}% (improved)")
        elif delta < 0:
            print(f"Overall change: {delta:.2f}% (decreased)")
        else:
            print("Overall change: 0.00% (no change)")

        length_delta = length_isolated - length_batch
        print(f"\nLength agreement (Batch):    {length_batch:.2f}%")
        print(f"Length agreement (Isolated): {length_isolated:.2f}%")
        if length_delta > 0:
            print(f"Length agreement improved by +{length_delta:.2f}%")
        elif length_delta < 0:
            print(f"Length agreement decreased by {length_delta:.2f}%")
        else:
            print("Length agreement unchanged.")


Running isolated text-based judging for 10 products x 5 criteria...
Checking Apple iPhone 15 Pro - fluency...
Checking Apple iPhone 15 Pro - grammar...
Checking Apple iPhone 15 Pro - tone...
Checking Apple iPhone 15 Pro - length...
Checking Apple iPhone 15 Pro - grounding...
Checking Samsung Galaxy S24 Ultra - fluency...
Checking Samsung Galaxy S24 Ultra - grammar...
Checking Samsung Galaxy S24 Ultra - tone...
Checking Samsung Galaxy S24 Ultra - length...
Checking Samsung Galaxy S24 Ultra - grounding...
Checking Google Pixel 8 Pro - fluency...
Checking Google Pixel 8 Pro - grammar...
Checking Google Pixel 8 Pro - tone...
Checking Google Pixel 8 Pro - length...
Checking Google Pixel 8 Pro - grounding...
Checking Sony WH‑1000XM5 Headphones - fluency...
Checking Sony WH‑1000XM5 Headphones - grammar...
Checking Sony WH‑1000XM5 Headphones - tone...
Checking Sony WH‑1000XM5 Headphones - length...
Checking Sony WH‑1000XM5 Headphones - grounding...
Checking Bose QuietComfort Ultra Earbuds - fl

,product_name,criterion,human_verdict,batch_verdict,isolated_verdict,batch_matches_human,isolated_matches_human
0,Apple iPhone 15 Pro,fluency,good,good,good,True,True
1,Apple iPhone 15 Pro,grammar,good,good,good,True,True
2,Apple iPhone 15 Pro,tone,good,ok,good,False,True
3,Apple iPhone 15 Pro,length,ok,good,good,False,False
4,Apple iPhone 15 Pro,grounding,good,good,good,True,True
5,Samsung Galaxy S24 Ultra,fluency,good,good,good,True,True
6,Samsung Galaxy S24 Ultra,grammar,good,good,good,True,True
7,Samsung Galaxy S24 Ultra,tone,good,good,good,True,True
8,Samsung Galaxy S24 Ultra,length,good,good,good,True,True
9,Samsung Galaxy S24 Ultra,grounding,good,good,good,True,True



Agreement Summary by Criterion


,criterion,batch_agreement_percent,isolated_agreement_percent
0,fluency,100.0,100.0
1,grammar,100.0,100.0
2,grounding,100.0,100.0
3,length,40.0,80.0
4,tone,90.0,100.0



Final Agreement Rates
Batch Judging vs Human:    86.00%
Isolated Judging vs Human: 96.00%
Overall change: +10.00% (improved)

Length agreement (Batch):    40.00%
Length agreement (Isolated): 80.00%
Length agreement improved by +40.00%


## Task 6.5: Summary and Strategic Analysis

### a) Practical Trade-offs: Human Evaluation vs. LLM-as-a-Judge

- **Cost**
  - **Human Evaluation**: High. Requires significant budget for manual labor and expert time.
  - **LLM-as-a-Judge**: Minimal. Extremely low API costs per product processed.

- **Scale**
  - **Human Evaluation**: Limited. Humans fatigue and struggle to evaluate thousands of items daily.
  - **LLM-as-a-Judge**: Effectively unlimited for this use case. Can process thousands of descriptions in minutes with parallel calls.

- **Consistency**
  - **Human Evaluation**: Subjective. Different annotators may score criteria like Tone or Fluency differently.
  - **LLM-as-a-Judge**: High consistency. Deterministic behavior at `temperature=0` applies the rubric uniformly.

- **Accuracy**
  - **Human Evaluation**: Gold standard for brand nuance, context, and subtle emotional framing.
  - **LLM-as-a-Judge**: Strong logic-based scoring (especially grammar/grounding), but objective criteria (like strict length checks) improve when judged in isolation.

**Observation from our experiment:**

Our run showed that while the LLM judge is extremely fast, accuracy on objective metrics (especially Length) improved significantly only after switching to **Isolated Judging** in Task 6.4. This indicates AI-as-a-judge is not fully plug-and-play and still requires careful prompt engineering for human-level precision.

### b) Production System Recommendation

For a production pipeline generating thousands of descriptions per day, I recommend a **Hybrid Human-in-the-Loop Model**:

- **AI as first-line quality gate**
  - Use LLM-as-a-judge on 100% of generated descriptions for scalable, immediate screening.

- **Task isolation for critical criteria**
  - Based on Task 6.4 findings, evaluate strict criteria (especially **Grounding** and **Length**) via isolated calls for stronger reliability.

- **Human spot-checking (QA)**
  - Route a random sample (e.g., 5%) of passed outputs to expert reviewers to prevent model drift and ensure brand alignment.

- **Threshold-based escalation**
  - Automatically flag any output with **OK** or **Bad** on critical criteria for quick human review before publication.

**Final Conclusion:**

This hybrid design combines AI speed and scalability with human judgment quality, producing a content workflow that is both operationally efficient and quality-safe.
